In [1]:
from datasets import load_dataset
from tqdm.notebook import tqdm

In [2]:
dataset = load_dataset('json', data_files='PyranetStatisticData/dataset_all_with_cell_num_error_timeout.jsonl', split = "train")
# dataset = load_dataset('bnadimi/PyraNet-Verilog', split = "train")

In [3]:
dataset

Dataset({
    features: ['description', 'code', 'module_definition', 'module_def_span', 'synth_status', 'num_cell'],
    num_rows: 661318
})

In [4]:
import numpy as np

all_cell_num = np.array(dataset['num_cell'])
all_cell_num

array([0.0, 2.0, 16.0, ..., None, None, None],
      shape=(661318,), dtype=object)

In [7]:
all_cell_num = np.column_stack((all_cell_num, range(len(dataset['num_cell']))))
all_cell_num

array([[0.0, 0],
       [2.0, 1],
       [16.0, 2],
       ...,
       [None, 661315],
       [None, 661316],
       [None, 661317]], shape=(661318, 2), dtype=object)

In [8]:
all_cell_num = np.column_stack((all_cell_num, [None] * len(dataset['num_cell'])))
all_cell_num

array([[0.0, 0, None],
       [2.0, 1, None],
       [16.0, 2, None],
       ...,
       [None, 661315, None],
       [None, 661316, None],
       [None, 661317, None]], shape=(661318, 3), dtype=object)

In [9]:
all_cell_num_no_none_idx = np.where(all_cell_num[:, 0] != None)
all_cell_num = all_cell_num[all_cell_num_no_none_idx]
all_cell_num

array([[0.0, 0, None],
       [2.0, 1, None],
       [16.0, 2, None],
       ...,
       [277.0, 661292, None],
       [3.0, 661309, None],
       [0.0, 661311, None]], shape=(354828, 3), dtype=object)

In [10]:
for i in range(len(all_cell_num)):
    int_x = all_cell_num[i][0]
    if int_x == 0:
        all_cell_num[i][2] = '0-0'
    else:
        # 1-100, 
        # 101-1000
        # 1001-10000
        # 10001-100000
        # 100001-25000

        start_i = 1
        stop_i = 100
        step_i = 10
        for ii in range(start_i,stop_i, step_i):
            cur_range = (ii, ii + step_i - 1)
            if cur_range[0] <= int_x <= cur_range[1]:
                all_cell_num[i][2] = f'{cur_range[0]}-{cur_range[1]}'
                break
all_cell_num

array([[0.0, 0, '0-0'],
       [2.0, 1, '1-10'],
       [16.0, 2, '11-20'],
       ...,
       [277.0, 661292, None],
       [3.0, 661309, '1-10'],
       [0.0, 661311, '0-0']], shape=(354828, 3), dtype=object)

In [11]:
all_range_no_none_idx = np.where(all_cell_num[:, 2] != None)
all_range_no_none_idx

(array([     0,      1,      2, ..., 354823, 354826, 354827],
       shape=(233720,)),)

In [12]:
dataset_select = dataset.select(all_cell_num[all_range_no_none_idx, 1][0])
dataset_select

Dataset({
    features: ['description', 'code', 'module_definition', 'module_def_span', 'synth_status', 'num_cell'],
    num_rows: 233720
})

In [139]:
import json, re, tempfile, subprocess, os
import numpy as np

def formatting_prompts_func(examples):
    # instructions = examples["instruction"]
    # inputs       = examples["input"]
    # outputs      = examples["output"]
    descriptions = examples["description"]
    module_def_spans = examples["module_def_span"]
    codes = examples["code"]
    chats = []
    for description, module_def_span, code in zip(descriptions, module_def_spans, codes):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        description = description.replace("\\\\", "\\")
        description = json.loads(description)['description']
        code = code.replace("\\\\", "\\")
        
        module_def_span = module_def_span
        module_defs = "\n".join([code[span[0]:span[2]] for span in module_def_span])
        
        instruction = "You are a Verilog designer. Generate appropriate Verilog code according to the description and Verilog module definition given below."
        input = description + "\n"*2 + module_defs
        output = code
        # print(text)
        chat = [
            # {"role": "system", "content": instruction},
            {"role": "user", "content": input},
            {"role": "assistant", "content": output}]
        chats.append(chat)
    return { "chat" : chats, }

# from datasets import load_dataset
# dataset = load_dataset("yahma/alpaca-cleaned", split = "train")
# dataset = dataset.map(formatting_prompts_func, batched = True,)